In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from datetime import timedelta
from hashlib import sha256
import json
from pathlib import Path
import re
import unicodedata

import lseg.data as ld
import pandas as pd


# ========= 設定 =========
@dataclass
class FetchConfig:
    query: str = "Language:LJA AND Source:RTRS"
    start: str = "2024-01-01T00:00:00Z"
    end: str = "2024-12-31T23:59:59Z"
    count: int = 100
    order_by: str = "new_to_old"
    chunk_days: int = 7
    cache_dir: str = "data/raw/headlines"
    force_refresh: bool = False


# ========= ヘルパー =========
def hash_values(values: list[str]) -> str:
    return sha256("||".join(values).encode("utf-8")).hexdigest()


def find_first_existing_col(df: pd.DataFrame, candidates: list[str]) -> str:
    for col in candidates:
        if col in df.columns:
            return col
    raise ValueError(f"候補カラムが見つかりません: {candidates}")


# ========= 正規化 =========
TIMESTAMP_CANDIDATES = ["versionCreated", "created", "published", "firstCreated", "date"]
HEADLINE_CANDIDATES = ["headline", "text", "storyHeadline", "headlineText", "title"]
STORY_ID_CANDIDATES = ["storyId", "story_id", "storyID", "id", "news_id"]
SOURCE_CANDIDATES = ["sourceCode", "source", "provider", "sourceName"]


def normalize_headlines_df(
    raw_df: pd.DataFrame,
    query: str = "",
    request_start: str | None = None,
    request_end: str | None = None,
    retrieved_at_utc: pd.Timestamp | None = None,
) -> pd.DataFrame:
    df = raw_df.copy()
    if df.empty:
        return pd.DataFrame(columns=[
            "news_id", "story_id", "timestamp", "headline", "source",
            "query", "request_start", "request_end", "retrieved_at_utc"
        ])

    # versionCreated index 優先
    if df.index.name == "versionCreated":
        df = df.reset_index()
        time_col = "versionCreated"
    elif isinstance(df.index, pd.DatetimeIndex):
        df = df.reset_index()
        time_col = str(df.columns[0])
    else:
        df = df.reset_index()
        time_col = "versionCreated" if "versionCreated" in df.columns else find_first_existing_col(df, TIMESTAMP_CANDIDATES)

    headline_col = find_first_existing_col(df, HEADLINE_CANDIDATES)
    story_col = find_first_existing_col(df, STORY_ID_CANDIDATES)
    source_col = find_first_existing_col(df, SOURCE_CANDIDATES)

    out = pd.DataFrame()
    out["story_id"] = df[story_col].fillna("").astype(str)
    out["headline"] = df[headline_col].fillna("").astype(str)
    out["source"] = df[source_col].fillna("").astype(str)
    out["timestamp"] = pd.to_datetime(df[time_col], errors="coerce", utc=True)
    out = out.dropna(subset=["timestamp"]).reset_index(drop=True)

    out["query"] = query
    out["request_start"] = request_start
    out["request_end"] = request_end
    out["retrieved_at_utc"] = pd.to_datetime(retrieved_at_utc or pd.Timestamp.now("UTC"), utc=True)

    out["news_id"] = out.apply(
        lambda r: hash_values([str(r["story_id"]), str(r["timestamp"]), str(r["headline"]), str(r["source"])])[:20],
        axis=1,
    )

    return out[[
        "news_id", "story_id", "timestamp", "headline", "source",
        "query", "request_start", "request_end", "retrieved_at_utc"
    ]]


# ========= 取得 =========
def _resolve_order_by(order_by: str):
    return getattr(ld.news.SortOrder, order_by) if hasattr(ld.news.SortOrder, order_by) else order_by


def _make_chunk_windows(start: pd.Timestamp, end: pd.Timestamp, chunk_days: int):
    windows = []
    cursor = start
    while cursor < end:
        nxt = min(cursor + timedelta(days=chunk_days), end)
        windows.append((cursor, nxt))
        cursor = nxt
    return windows


def fetch_headlines(cfg: FetchConfig) -> tuple[pd.DataFrame, pd.DataFrame]:
    cache_dir = Path(cfg.cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)

    start = pd.to_datetime(cfg.start, utc=True)
    end = pd.to_datetime(cfg.end, utc=True)
    order_by = _resolve_order_by(cfg.order_by)

    raw_frames = []
    norm_frames = []

    for chunk_start, chunk_end in _make_chunk_windows(start, end, cfg.chunk_days):
        chunk_key = hash_values([cfg.query, chunk_start.isoformat(), chunk_end.isoformat(), str(cfg.count), cfg.order_by])[:16]
        raw_path = cache_dir / f"headlines_raw_{chunk_key}.parquet"
        meta_path = cache_dir / f"headlines_raw_{chunk_key}.json"

        if raw_path.exists() and not cfg.force_refresh:
            chunk_raw = pd.read_parquet(raw_path)
            if "_index_versionCreated" in chunk_raw.columns:
                chunk_raw = chunk_raw.set_index("_index_versionCreated")
                chunk_raw.index.name = "versionCreated"
        else:
            chunk_raw = ld.news.get_headlines(
                query=cfg.query,
                count=int(cfg.count),
                start=chunk_start.to_pydatetime(),
                end=chunk_end.to_pydatetime(),
                order_by=order_by,
            )
            to_save = chunk_raw.copy()
            if isinstance(to_save.index, pd.DatetimeIndex) or to_save.index.name == "versionCreated":
                index_name = to_save.index.name or "index"
                to_save = to_save.reset_index().rename(columns={index_name: "_index_versionCreated"})
            to_save.to_parquet(raw_path, index=False)
            meta_path.write_text(json.dumps({
                "query": cfg.query, "start": chunk_start.isoformat(), "end": chunk_end.isoformat(),
                "count": cfg.count, "order_by": cfg.order_by
            }, ensure_ascii=False, indent=2), encoding="utf-8")

        raw_frames.append(chunk_raw)
        norm_frames.append(normalize_headlines_df(
            chunk_raw,
            query=cfg.query,
            request_start=chunk_start.isoformat(),
            request_end=chunk_end.isoformat(),
            retrieved_at_utc=pd.Timestamp.now("UTC"),
        ))

    raw_all = pd.concat(raw_frames, axis=0) if raw_frames else pd.DataFrame()
    norm_all = pd.concat(norm_frames, ignore_index=True) if norm_frames else normalize_headlines_df(pd.DataFrame())
    if not norm_all.empty:
        norm_all = norm_all.sort_values("timestamp").drop_duplicates(subset=["news_id"], keep="first").reset_index(drop=True)

    return raw_all, norm_all


# ========= 前処理 =========
PREFIXES = [
    "UPDATE 1-", "UPDATE 2-", "UPDATE 3-", "RPT-", "BREAKING-", "BREAKINGVIEWS-",
    "BUZZ-", "訂正-", "再送-", "焦点：", "アングル：", "コラム：", "インタビュー：",
    "〔マーケットアイ〕", "〔需給情報〕",
]
LOW_INFORMATION_PATTERNS = ["東証前引け", "東証大引け", "今日の株式見通し", "午前の日経平均", "午後の日経平均", "外為市場", "需給情報"]
_URL_RE = re.compile(r"https?://\S+|www\.\S+", flags=re.IGNORECASE)
_SPACE_RE = re.compile(r"\s+")


def _strip_prefixes(text: str) -> str:
    out = text
    while True:
        changed = False
        for prefix in PREFIXES:
            if out.startswith(prefix):
                out = out[len(prefix):].lstrip(" -:：")
                changed = True
        if not changed:
            break
    return out


def _normalize_headline(text: str) -> str:
    out = unicodedata.normalize("NFKC", text)
    out = _URL_RE.sub(" ", out)
    out = _strip_prefixes(out)
    out = _SPACE_RE.sub(" ", out)
    return out.strip()


def _is_low_information(text: str) -> bool:
    return any(p in text for p in LOW_INFORMATION_PATTERNS)


def clean_headlines(df: pd.DataFrame, mode: str = "drop", min_chars: int = 8) -> pd.DataFrame:
    if mode not in {"drop", "flag"}:
        raise ValueError("mode は 'drop' または 'flag'")

    out = df.copy()
    out["headline"] = out["headline"].fillna("").astype(str)
    out["headline_clean"] = out["headline"].map(_normalize_headline)
    out["low_information_flag"] = out["headline_clean"].map(_is_low_information)
    out["too_short_flag"] = out["headline_clean"].str.len() < int(min_chars)

    out = out[~out["too_short_flag"]].copy()
    if mode == "drop":
        out = out[~out["low_information_flag"]].copy()
    return out.reset_index(drop=True)

ld.open_session()
cfg = FetchConfig()
raw_df, normalized_df = fetch_headlines(cfg)
cleaned_df = clean_headlines(normalized_df, mode="drop", min_chars=8)

In [ ]:

from .embeddings import encode_texts_with_cache
from .dedup import deduplicate_headlines

cache_path = cfg.resolve_path(cfg.embedding.headline_cache_path)
assert cache_path is not None

# 1) headline_clean を embedding 化（キャッシュ再利用）
emb_meta, all_embeddings = encode_texts_with_cache(
    ids=clean_df["news_id"],
    texts=clean_df["headline_clean"],
    cache_path=cache_path,
    model_name=cfg.embedding.model_name,
    batch_size=cfg.embedding.batch_size,
    normalize_embeddings=cfg.embedding.normalize_embeddings,
)

# 2) 重複抑制
dedup_df, dedup_log = deduplicate_headlines(
    clean_df,
    embeddings=all_embeddings,
    similarity_threshold=cfg.dedup.similarity_threshold,  # 既定 0.92
    window_hours=cfg.dedup.window_hours,                  # 既定 24h
)


In [ ]:
# pipeline.py 抜粋（重複除去の次）

def _select_fit_sample(df: pd.DataFrame, cfg: PipelineConfig) -> pd.DataFrame:
    out = df.copy()
    out["timestamp"] = pd.to_datetime(out["timestamp"], utc=True, errors="coerce")

    fit_start = cfg.topic_model.fit_start
    fit_end = cfg.topic_model.fit_end

    if fit_start:
        out = out[out["timestamp"] >= pd.to_datetime(fit_start, utc=True)]
    if fit_end:
        out = out[out["timestamp"] <= pd.to_datetime(fit_end, utc=True)]

    if out.empty:
        raise ValueError("BERTopic学習期間に該当するニュースがありません。")
    return out.reset_index(drop=True)

# dedup済みニュースに対応する embedding を再取得
clean_pos = clean_df.reset_index()[["index", "news_id"]]
dedup_indexer = (
    dedup_df[["news_id"]]
    .merge(clean_pos, on="news_id", how="left")["index"]
    .to_numpy(dtype=int)
)
dedup_emb = all_embeddings[dedup_indexer]

# fit対象期間を抽出し、対応embeddingを取得
fit_df = _select_fit_sample(dedup_df, cfg)
dedup_pos = dedup_df.reset_index()[["index", "news_id"]]
fit_idx = (
    fit_df[["news_id"]]
    .merge(dedup_pos, on="news_id", how="left")["index"]
    .to_numpy(dtype=int)
)
fit_emb = dedup_emb[fit_idx]

# BERTopic fit -> full transform
topic_model, fit_tables = fit_topic_model(fit_df, fit_emb, cfg.topic_model)
topic_assignments = transform_topics(dedup_df, topic_model, dedup_emb)
topic_tables = build_topic_tables(topic_assignments, topic_model, cfg.topic_model.top_n_headlines)


In [ ]:
# pipeline.py 抜粋（BERTopic付与の直後）

daily_intensity, weekly_intensity, outlier_stats = build_topic_intensity(
    topic_assignments,
    weekly_rule=cfg.topic_intensity.weekly_rule,   # "W-FRI"
    ewma_span=cfg.topic_intensity.ewma_span,       # 5
    lookback=cfg.topic_intensity.lookback,         # 20
    z_threshold=cfg.topic_intensity.z_threshold,   # 1.0
    aggregate_timezone=cfg.topic_intensity.aggregate_timezone,  # "Asia/Tokyo"
)


In [ ]:
# pipeline.py 抜粋（topic-theme similarity -> exposure）
from pathlib import Path
import sys
import pandas as pd

PROJECT_SRC = Path("/Users/kencharoff/workspace/projects/thematic-topic/src")
if str(PROJECT_SRC) not in sys.path:
    sys.path.append(str(PROJECT_SRC))

from thematic_topic.data_io import load_theme_definitions
from thematic_topic.theme import build_theme_profiles, map_topics_to_themes
from thematic_topic.exposure import build_exposure

# Notebook用ミニ設定
cfg_topic_theme = {
    "theme_definitions_path": "/Users/kencharoff/Downloads/theme_profiles.yaml",
    "model_name": "paraphrase-multilingual-MiniLM-L12-v2",
    "similarity_threshold": 0.35,
    "top_n_themes_per_topic": 5,
    "exposure_intensity_col": "topic_ewma",
}
save_outputs = False
output_dir = Path("/Users/kencharoff/workspace/projects/thematic-topic/outputs/tables")

print("=== topic-theme/exposure config ===")
display(pd.DataFrame([cfg_topic_theme]))
print(f"save_outputs={save_outputs}")

# 入力依存チェック
if "daily_intensity" not in globals():
    raise ValueError("daily_intensity が見つかりません。先に topic強度時系列セルを実行してください。")

if "topic_tables" in globals() and isinstance(topic_tables, dict) and "topic_summary" in topic_tables:
    topic_summary_df = topic_tables["topic_summary"]
elif "topic_summary" in globals():
    topic_summary_df = topic_summary
else:
    raise ValueError("topic_summary_df が見つかりません。topic_tables['topic_summary'] または topic_summary を先に生成してください。")

# テーマ定義（YAML） -> 既存必須スキーマへ正規化
theme_df = load_theme_definitions(Path(cfg_topic_theme["theme_definitions_path"]))
theme_profiles = build_theme_profiles(theme_df)

# topic-theme 類似度マッピング
topic_theme_map = map_topics_to_themes(
    topic_summary_df=topic_summary_df,
    theme_profile_df=theme_profiles,
    model_name=cfg_topic_theme["model_name"],
    similarity_threshold=cfg_topic_theme["similarity_threshold"],
    top_n_themes_per_topic=cfg_topic_theme["top_n_themes_per_topic"],
)

print("=== topic_theme_map summary ===")
print(f"rows={len(topic_theme_map):,}, topics={topic_theme_map['topic_id'].nunique():,}, themes={topic_theme_map['theme_id'].nunique():,}")
display(topic_theme_map.head(20))

# exposure 構築: X_{t,g,k} = topic_intensity * topic_theme_similarity
exposure_df = build_exposure(
    topic_intensity_df=daily_intensity,
    topic_theme_map_df=topic_theme_map,
    intensity_col=cfg_topic_theme["exposure_intensity_col"],
)

print("=== exposure summary ===")
if exposure_df.empty:
    print("rows=0 (該当データなし)")
else:
    d = pd.to_datetime(exposure_df["date"], errors="coerce")
    print(
        f"rows={len(exposure_df):,}, date_range={d.min()} -> {d.max()}, "
        f"themes={exposure_df['theme_id'].nunique():,}, topics={exposure_df['topic_id'].nunique():,}"
    )
display(exposure_df.head(20))

if save_outputs:
    output_dir.mkdir(parents=True, exist_ok=True)
    topic_theme_map.to_parquet(output_dir / "topic_theme_map.parquet", index=False)
    exposure_df.to_parquet(output_dir / "theme_topic_exposure_daily.parquet", index=False)
    print(f"saved: {output_dir / 'topic_theme_map.parquet'}")
    print(f"saved: {output_dir / 'theme_topic_exposure_daily.parquet'}")

